# Lab: Context 전략 실습 과제 (YOU DO)

**과제**: 자신의 도메인 태스크에 다양한 컨텍스트 전략을 적용하고, 결과를 비교하라.

**요구사항:**
1. 도메인 태스크를 하나 선택하라 (예: 기술 문서 분류, 버그 우선순위 판단 등)
2. 3가지 이상의 컨텍스트 전략을 적용하라
3. 각 전략의 토큰 수, 비용, 응답 품질을 비교하라

**정답**: `solution/context_strategies.ipynb` 참고

> **Jupyter 노트북 실행 참고**: `agent.run_sync()` 대신 `await agent.run()`을 사용한다.
> Jupyter는 이미 이벤트 루프가 실행 중이므로 `run_sync()`는 `RuntimeError`를 일으킨다.

## 환경 설정

필요한 라이브러리를 임포트하고 `.env`에서 API 키를 로드한다.

In [62]:
import time
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from enum import Enum
from dotenv import load_dotenv

# .env 파일에서 OPENAI_API_KEY 등 환경변수를 로드
load_dotenv()

True

## 모델 및 비용 설정

비용 계산 함수를 정의한다. 입력/출력 토큰 단가가 다르므로 분리하여 계산한다.

In [63]:
# 사용할 모델 지정
MODEL = "openai:gpt-5.4"

# 비용 단가 ($/1M tokens)
PRICING = {
    "openai:gpt-5.4": {"input": 2.50, "output": 10.00},
    "openai:gpt-5.4-mini": {"input": 0.15, "output": 0.60},
}


def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """토큰 수와 단가로 비용(USD)을 계산한다"""
    prices = PRICING.get(model, {"input": 2.50, "output": 10.00})
    return (input_tokens * prices["input"] + output_tokens * prices["output"]) / 1_000_000

---
## TODO 1: Pydantic 스키마 정의

자신의 도메인에 맞는 분류 스키마를 정의하라.

예시: 기술 블로그 카테고리 분류, 버그 우선순위 판단, 이메일 의도 분류 등

**힌트:**
- `Enum`으로 카테고리를 제한하면 LLM이 범위 밖의 값을 생성할 수 없다
- `Field`의 `description`은 LLM에게 각 필드의 의미를 알려주는 힌트다
- PydanticAI의 `output_type`에 이 모델을 전달하면 스키마가 자동으로 강제된다

```python
class Category(str, Enum):
    BUG = "버그"
    FEATURE = "기능요청"
    QUESTION = "사용법문의"
```

In [64]:
# TODO: 아래 스키마를 자신의 도메인에 맞게 수정하라
class Category(str, Enum):
    """회의 일정 조율 요청 종류"""
    NEW_MEETING = "새 일정 요청"
    RESCHEDULE = "재조정 요청"
    CONFLICT = "시간 충돌 해결"


class Priority(str, Enum):
    """일정 조율 우선순위"""
    HIGH = "높음"
    MEDIUM = "보통"
    LOW = "낮음"


# TODO: 필드의 description을 도메인에 맞게 작성하라
# description은 LLM이 참조하는 힌트 — 명확할수록 결과가 좋다
class AnalysisResult(BaseModel):
    """회의 일정 자동 조율 분석 결과"""
    category: Category = Field(
        description=(
            "회의 일정 자동 조율 문맥에서 요청 유형을 분류합니다."
            " 예: 새 일정 요청, 재조정 요청, 시간 충돌 해결"
        )
    )
    priority: Priority = Field(
        description=(
            "일정 조율의 긴급도를 나타냅니다."
            " 고객사 미팅, 불변 일정 포함 요청 등은 높음입니다."
        )
    )
    summary: str = Field(
        description=(
            "요청의 핵심을 한 문장으로 요약합니다."
            " 예: '고객사 미팅 불변 일정 포함, 대안 시간 제안 필요'"
        )
    )

---
## TODO 2: 테스트 데이터 정의

자신의 도메인에서 5개 이상의 테스트 케이스를 만들어라.

`(입력 텍스트, 기대 카테고리, 기대 우선순위)` 형태로 작성한다.
쉬운 케이스부터 경계 케이스(모호한 입력)까지 다양하게 구성하면 전략 간 차이가 잘 드러난다.

In [65]:
# TODO: 테스트 케이스를 5개 이상 작성하라
# 각 튜플: (입력 텍스트, 기대 카테고리, 기대 우선순위)
TEST_CASES = [
    (
        "내일 오전 10시에 고객사 미팅을 잡고 싶습니다. 팀원 전체의 참석 가능 여부를 확인하고 가능한 시간대를 추천해주세요.",
        "새 일정 요청",
        "높음",
    ),
    (
        "금요일 오후 3시 일정이 있는데 참석자 중 한 명이 일정 변경을 요청했습니다. 대체 가능한 시간대를 찾아서 제안해주세요.",
        "재조정 요청",
        "보통",
    ),
    (
        "고객사 미팅은 변경 불가 일정입니다. 이 시간대를 제외하고 모든 참석자의 빈 시간대 교집합을 찾아주세요.",
        "시간 충돌 해결",
        "높음",
    ),
    (
        "팀즈에서 수집한 참석자 일정 상 오후 2시 이후에는 모두 불가능합니다. 가능한 대체 시간을 제안해야 합니다.",
        "시간 충돌 해결",
        "보통",
    ),
    (
        "내부 미팅을 새로 잡으려고 합니다. 우선순위가 낮고 일주일 내로 가능하면 좋겠습니다.",
        "새 일정 요청",
        "낮음",
    ),
]

---
## TODO 3: 전략 A - System Prompt 없이 호출

System Prompt(`instructions`) 없이 User 메시지만으로 분류를 요청한다.
`output_type` 덕분에 스키마는 강제되지만, 분류 판단은 LLM의 일반 지식에 의존한다.

**힌트:**
```python
async def strategy_no_system(text: str) -> tuple[AnalysisResult, dict]:
    agent = Agent(MODEL, output_type=AnalysisResult)
    result = await agent.run(f"다음 텍스트를 분류하라:\n\n{text}")
    usage = result.usage()
    ...
```

> **주의**: Jupyter에서는 `run_sync()` 대신 `await agent.run()`을 사용한다.

In [66]:
async def strategy_no_system(text: str) -> tuple[AnalysisResult, dict]:
    """System Prompt(instructions) 없이 User 메시지만으로 분류를 요청한다.

    반환: (분석결과, 메타데이터 dict)
    메타데이터: {"input_tokens": ..., "output_tokens": ..., "latency": ..., "cost": ...}
    """
    start = time.time()

    # TODO: instructions 없이 Agent를 생성하고 await agent.run()으로 호출하라
    agent = Agent(MODEL, output_type=AnalysisResult)
    result = await agent.run(
        "다음 텍스트를 읽고 회의 일정 자동 조율 도메인에서 요청 유형, 우선순위, 요약을 반환하세요:\n\n"
        + text
    )

    usage = result.usage()
    input_tokens = getattr(usage, "input_tokens", getattr(usage, "prompt_tokens", 0))
    output_tokens = getattr(usage, "output_tokens", getattr(usage, "completion_tokens", 0))
    latency = time.time() - start
    cost = calculate_cost(MODEL, input_tokens, output_tokens)

    return result.output, {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
        "cost": cost,
    }

---
## TODO 4: 전략 B - 역할 + 제약이 있는 System Prompt

역할 지정 + 분류 기준이 포함된 `instructions`로 호출한다.
분류 판단의 근거를 LLM에게 제공하여 정확도를 높인다.

**힌트:**
```python
async def strategy_with_constraints(text: str) -> tuple[AnalysisResult, dict]:
    agent = Agent(
        MODEL,
        instructions=(
            "당신은 {도메인} 분석 전문가입니다.\n\n"
            "카테고리 기준:\n"
            "- 카테고리A: ...\n"
            "- 카테고리B: ...\n"
        ),
        output_type=AnalysisResult,
    )
    result = await agent.run(text)
```

In [67]:
async def strategy_with_constraints(text: str) -> tuple[AnalysisResult, dict]:
    """역할 지정 + 분류 기준이 포함된 instructions로 호출한다.

    반환: (분석결과, 메타데이터 dict)
    """
    start = time.time()

    # TODO: 역할과 제약 조건이 포함된 instructions로 Agent를 생성하라
    # TODO: await agent.run(text)로 호출하라
    agent = Agent(
        MODEL,
        instructions=(
            "당신은 회의 일정 자동 조율 전문가입니다.\n"
            "아래 텍스트를 분석하여 요청 유형, 우선순위, 요약을 반환하세요.\n\n"
            "카테고리 기준:\n"
            "- 새 일정 요청: 참석자들의 가능한 시간대를 조사하고 새 일정을 잡아야 하는 경우\n"
            "- 재조정 요청: 기존 일정 중 일부를 변경하거나 대체 시간을 찾는 경우\n"
            "- 시간 충돌 해결: 불변 일정이 포함되어 교집합을 찾거나 일부 참석자를 제외해야 할 가능성이 있는 경우\n\n"
            "우선순위 기준:\n"
            "- 높음: 고객사 미팅, 당일 일정, 변경 불가 일정이 포함된 경우\n"
            "- 보통: 일주일 안에 조율해야 하며 대체 시간 제안이 필요한 경우\n"
            "- 낮음: 일정 여유가 있고 우선순위가 낮은 내부 회의인 경우\n\n"
            "요약에는 추천 다음 단계와 핵심 제약을 간결하게 작성하세요."
        ),
        output_type=AnalysisResult,
    )
    result = await agent.run(text)

    usage = result.usage()
    input_tokens = getattr(usage, "input_tokens", getattr(usage, "prompt_tokens", 0))
    output_tokens = getattr(usage, "output_tokens", getattr(usage, "completion_tokens", 0))
    latency = time.time() - start
    cost = calculate_cost(MODEL, input_tokens, output_tokens)

    return result.output, {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
        "cost": cost,
    }

---
## TODO 5: 전략 C - Few-shot 예시 포함

Few-shot 예시를 `instructions`에 포함하여 호출한다.
경계 케이스(모호한 입력)를 예시에 포함하면 정확도가 크게 향상된다.

**힌트:**
```python
async def strategy_few_shot(text: str) -> tuple[AnalysisResult, dict]:
    agent = Agent(
        MODEL,
        instructions=(
            "당신은 {도메인} 분석 전문가입니다.\n\n"
            "예시 1:\n"
            "입력: ...\n"
            "분류: 카테고리=..., 우선순위=...\n\n"
            "예시 2:\n"
            "입력: ...\n"
            "분류: 카테고리=..., 우선순위=...\n"
        ),
        output_type=AnalysisResult,
    )
    result = await agent.run(text)
```

> Few-shot은 토큰 사용량이 가장 크지만, 모호한 입력에서 정확도가 높다.

In [68]:
async def strategy_few_shot(text: str) -> tuple[AnalysisResult, dict]:
    """Few-shot 예시를 instructions에 포함하여 호출한다.

    반환: (분석결과, 메타데이터 dict)
    """
    start = time.time()

    # TODO: Few-shot 예시 2-3개를 instructions에 instructions를 포함하여 Agent를 생성하라
    # TODO: await agent.run(text)로 호출하라
    agent = Agent(
        MODEL,
        instructions=(
            "당신은 회의 일정 자동 조율 전문가입니다.\n"
            "입력과 출력 예시를 참고하여 요청 유형, 우선순위, 요약을 반환하세요.\n\n"
            "예시 1:\n"
            "입력: 내일 오전 10시에 고객사 미팅을 잡고 싶습니다. 팀원 전체의 참석 가능 여부를 확인하고 가능한 시간대를 추천해주세요.\n"
            "분류: category=새 일정 요청, priority=높음, summary=고객사 미팅 요청으로 가능한 시간대 추천 필요\n\n"
            "예시 2:\n"
            "입력: 금요일 오후 3시 일정이 있는데 참석자 중 한 명이 일정 변경을 요청했습니다. 대체 가능한 시간대를 찾아서 제안해주세요.\n"
            "분류: category=재조정 요청, priority=보통, summary=기존 일정 재조정 요청, 대체 시간 제안 필요\n\n"
            "예시 3:\n"
            "입력: 고객사 미팅은 변경 불가 일정입니다. 이 시간대를 제외하고 모든 참석자의 빈 시간대 교집합을 찾아주세요.\n"
            "분류: category=시간 충돌 해결, priority=높음, summary=불변 일정 포함, 교집합 가능한 시간대 탐색 필요\n"
        ),
        output_type=AnalysisResult,
    )
    result = await agent.run(text)

    usage = result.usage()
    input_tokens = getattr(usage, "input_tokens", getattr(usage, "prompt_tokens", 0))
    output_tokens = getattr(usage, "output_tokens", getattr(usage, "completion_tokens", 0))
    latency = time.time() - start
    cost = calculate_cost(MODEL, input_tokens, output_tokens)


    return result.output, {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
        "cost": cost,
    }

---
## 비교 실행

3가지 전략을 동일한 테스트 케이스에 적용하고 비교 리포트를 출력한다.

전략 함수들이 `async def`이므로 비교 함수도 `async def`로 정의하고 `await`로 호출한다.
**(수정 금지)**

In [69]:
async def run_comparison():
    """3가지 전략을 동일한 테스트 케이스에 적용하고 비교 리포트를 출력한다"""
    strategies = [
        ("A. System 없음", strategy_no_system),
        ("B. 역할 + 제약", strategy_with_constraints),
        ("C. Few-shot", strategy_few_shot),
    ]

    if not TEST_CASES:
        print("[오류] TEST_CASES가 비어 있습니다. TODO 2를 완성하세요.")
        return

    print("Context 전략 비교 결과")
    print("=" * 70)

    summary_rows = []

    for strategy_name, strategy_fn in strategies:
        print(f"\n[{strategy_name}]")
        print(f"{'─'*70}")

        correct = 0
        total_cost = 0.0
        total_latency = 0.0
        total_input_tokens = 0
        total_output_tokens = 0

        for i, (text, expected_cat, expected_pri) in enumerate(TEST_CASES, 1):
            try:
                # 전략 함수가 async이므로 await로 호출
                result, meta = await strategy_fn(text)

                cat_ok = result.category.value == expected_cat
                pri_ok = result.priority.value == expected_pri
                is_correct = cat_ok and pri_ok
                if is_correct:
                    correct += 1

                total_cost += meta["cost"]
                total_latency += meta["latency"]
                total_input_tokens += meta["input_tokens"]
                total_output_tokens += meta["output_tokens"]

                mark = "O" if is_correct else "X"
                print(
                    f"  {i}. 예측: {result.category.value}/{result.priority.value}, "
                    f"정답: {expected_cat}/{expected_pri} -> {mark}"
                )
            except NotImplementedError:
                print(f"  {i}. [미구현] 이 전략을 구현하세요.")
                break
            except Exception as e:
                print(f"  {i}. [오류] {e}")

        if correct > 0 or total_cost > 0:
            accuracy = correct / len(TEST_CASES) * 100
            print(f"\n  정확도:     {correct}/{len(TEST_CASES)} ({accuracy:.0f}%)")
            print(f"  총 비용:     ${total_cost:.6f}")
            print(f"  총 지연:     {total_latency:.2f}초")
            print(f"  입력 토큰:   {total_input_tokens}")
            print(f"  출력 토큰:   {total_output_tokens}")

            summary_rows.append({
                "strategy": strategy_name,
                "accuracy": accuracy,
                "cost": total_cost,
                "latency": total_latency,
                "input_tokens": total_input_tokens,
            })

    # 최종 비교 테이블
    if summary_rows:
        print(f"\n{'전략':<20} {'정확도':>8} {'비용($)':>10} {'지연(초)':>8} {'입력토큰':>8}")
        print(f"{'─'*60}")
        for row in summary_rows:
            print(
                f"{row['strategy']:<20} {row['accuracy']:>7.0f}% "
                f"{row['cost']:>10.6f} {row['latency']:>8.2f} "
                f"{row['input_tokens']:>8}"
            )

    print(f"\nLangSmith 대시보드에서 각 전략의 트레이스를 비교하세요!")
    print("https://smith.langchain.com")


# async 함수이므로 await로 호출
await run_comparison()

Context 전략 비교 결과

[A. System 없음]
──────────────────────────────────────────────────────────────────────
  1. 예측: 새 일정 요청/높음, 정답: 새 일정 요청/높음 -> O
  2. 예측: 재조정 요청/보통, 정답: 재조정 요청/보통 -> O
  3. 예측: 새 일정 요청/높음, 정답: 시간 충돌 해결/높음 -> X
  4. 예측: 시간 충돌 해결/보통, 정답: 시간 충돌 해결/보통 -> O
  5. 예측: 새 일정 요청/낮음, 정답: 새 일정 요청/낮음 -> O

  정확도:     4/5 (80%)
  총 비용:     $0.007157
  총 지연:     8.54초
  입력 토큰:   1703
  출력 토큰:   290

[B. 역할 + 제약]
──────────────────────────────────────────────────────────────────────
  1. 예측: 새 일정 요청/높음, 정답: 새 일정 요청/높음 -> O
  2. 예측: 재조정 요청/보통, 정답: 재조정 요청/보통 -> O
  3. 예측: 시간 충돌 해결/높음, 정답: 시간 충돌 해결/높음 -> O
  4. 예측: 재조정 요청/보통, 정답: 시간 충돌 해결/보통 -> X
  5. 예측: 새 일정 요청/낮음, 정답: 새 일정 요청/낮음 -> O

  정확도:     4/5 (80%)
  총 비용:     $0.010040
  총 지연:     8.78초
  입력 토큰:   2568
  출력 토큰:   362

[C. Few-shot]
──────────────────────────────────────────────────────────────────────
  1. 예측: 새 일정 요청/높음, 정답: 새 일정 요청/높음 -> O
  2. 예측: 재조정 요청/보통, 정답: 재조정 요청/보통 -> O
  3. 예측: 시간 충돌 해결/높음, 정답: 시간 충돌 해결/높음 -> O
  4. 